# 🧠 Steering Diffusion Language Models via SAE Feature Intervention

**Extending DLM-Scope (ICLR 2026) to steer DLMs toward chain-of-thought reasoning**

This notebook runs the complete experiment pipeline on Google Colab (Free T4 GPU):
1. ⚙️ Setup & Model Loading
2. 📊 Activation Collection (CoT vs Direct prompts on GSM8K)
3. 🔧 SAE Training (Top-K, DLM-Scope architecture)
4. 🔬 Contrastive Reasoning Feature Discovery
5. 🎯 Steering Experiments
6. 📈 Visualization & Results

**Checkpoints are saved to Google Drive** — you can resume from any phase if your session disconnects.

## Phase 0: Environment Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q transformers accelerate datasets scipy seaborn einops tqdm pyyaml

In [ ]:
# Mount Google Drive for checkpointing
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/dlm_steering'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoint directory: {SAVE_DIR}")

In [ ]:
# Clone the project repo
# ⚠️ REPLACE with your actual GitHub repo URL
REPO_URL = "https://github.com/YOUR_USERNAME/dlm-reasoning-steering.git"

if not os.path.exists('/content/dlm-reasoning-steering'):
    !git clone {REPO_URL} /content/dlm-reasoning-steering

%cd /content/dlm-reasoning-steering/project3_dlm_steering

# Add project to Python path
import sys
sys.path.insert(0, '/content/dlm-reasoning-steering/project3_dlm_steering')

## Phase 1: Load DiffuGPT-Medium & Verify

In [ ]:
import torch
import json
import logging
logging.basicConfig(level=logging.INFO)

from src.models.dlm_wrapper import DLMWrapper

print("Loading DiffuGPT-Medium (355M params)...")
dlm = DLMWrapper(
    model_name="diffusionfamily/diffugpt-m",
    base_model_name="gpt2-medium",
    quantize_4bit=False,
    cache_dir=os.path.join(SAVE_DIR, 'model_cache'),
)

print(f"\n✅ Model loaded: {dlm.get_model_info()}")

In [ ]:
# Quick generation test
print("🧪 Generation test:")
test_prompt = "The capital of France is"
output = dlm.generate(test_prompt, max_new_tokens=32, num_steps=16, temperature=0.8)
print(f"  Input:  '{test_prompt}'")
print(f"  Output: '{output}'")

# Activation extraction test
print("\n🧪 Activation extraction test:")
dlm.register_activation_hooks([4, 12, 20])
test_ids = dlm.tokenizer.encode(test_prompt, return_tensors='pt')
_ = dlm.forward_pass(test_ids)
acts = dlm.get_activations()
for layer_idx, act_tensor in acts.items():
    print(f"  Layer {layer_idx}: shape={act_tensor.shape}")
dlm.clear_hooks()

print("\n✅ Phase 1 complete")

## Phase 2: Collect Activations from GSM8K

We collect residual-stream activations under two conditions:
- **CoT prompts**: "Solve step by step: {question}" → encourages reasoning
- **Direct prompts**: "Answer directly: {question}" → encourages direct response

The difference in SAE feature activations between these conditions reveals **reasoning-associated features**.

In [ ]:
from src.data.gsm8k_loader import GSM8KLoader
from src.data.activation_collector import ActivationCollector

# Load GSM8K
print("Loading GSM8K...")
gsm8k = GSM8KLoader(split='test', n_problems=200, cache_dir=SAVE_DIR)

# Split into discovery (feature ID) and evaluation (steering test)
discovery_set, eval_set = gsm8k.split_discovery_eval(discovery_frac=0.5)
print(f"  Discovery: {len(discovery_set)} problems")
print(f"  Evaluation: {len(eval_set)} problems")

# Show a sample pair
cot = discovery_set.get_cot_prompts()[0]
direct = discovery_set.get_direct_prompts()[0]
print(f"\n--- Sample CoT Prompt ---\n{cot['prompt'][:200]}")
print(f"\n--- Sample Direct Prompt ---\n{direct['prompt'][:200]}")

In [ ]:
# Collect activations (this takes ~30-60 min on T4)
# Checkpoints to Drive after each batch - safe to resume!

TARGET_LAYERS = [4, 8, 12, 16, 20]  # Spanning DiffuGPT's 24 layers
TIMESTEP_SAMPLES = [0, 7, 14, 21, 29]  # 5 evenly-spaced from 30 steps

collector = ActivationCollector(
    dlm_wrapper=dlm,
    target_layers=TARGET_LAYERS,
    save_dir=os.path.join(SAVE_DIR, 'activations'),
    batch_size=2,
    denoising_steps=30,
    timestep_samples=TIMESTEP_SAMPLES,
)

# Collect CoT activations
print("📊 Collecting CoT activations...")
cot_prompts = discovery_set.get_cot_prompts()
cot_stats = collector.collect_from_prompts(
    prompts=cot_prompts,
    max_gen_tokens=128,
    collection_name='cot_activations',
    resume=True,
)
print(f"  ✅ CoT: {cot_stats['n_prompts']} prompts")

# Collect Direct activations
print("\n📊 Collecting Direct activations...")
direct_prompts = discovery_set.get_direct_prompts()
direct_stats = collector.collect_from_prompts(
    prompts=direct_prompts,
    max_gen_tokens=128,
    collection_name='direct_activations',
    resume=True,
)
print(f"  ✅ Direct: {direct_stats['n_prompts']} prompts")

## Phase 3: Train Top-K SAE

We train a Top-K SAE on the collected activations, following the DLM-Scope architecture:
- **Dictionary size**: 4× model dimension (4096 features for d_model=1024)
- **K**: 64 (Top-64 sparsity)
- **Training**: MSE reconstruction loss with decoder normalization

In [ ]:
from src.training.sae_trainer import train_sae_for_layer

PRIMARY_LAYER = 16  # Deep layer — DLM-Scope shows these are best for steering

# Load training activations
print(f"Loading activations for layer {PRIMARY_LAYER}...")
train_data = collector.get_training_data(
    collection_name='cot_activations',
    layer_idx=PRIMARY_LAYER,
    max_tokens=500_000,
)
print(f"  Training data: {train_data.shape}")

# Train SAE
print(f"\n🔧 Training SAE (this takes ~15-30 min)...")
sae = train_sae_for_layer(
    activations=train_data,
    d_model=dlm.d_model,
    layer_idx=PRIMARY_LAYER,
    save_dir=os.path.join(SAVE_DIR, 'saes'),
    expansion_factor=4,
    k=64,
    n_epochs=5,
)
print(f"\n✅ SAE trained: {sae}")

In [ ]:
# Evaluate SAE quality
from src.training.sae_trainer import SAETrainer

eval_data = collector.get_training_data(
    collection_name='direct_activations',
    layer_idx=PRIMARY_LAYER,
    max_tokens=50_000,
)

trainer = SAETrainer(d_model=dlm.d_model, expansion_factor=4, k=64)
trainer.sae = sae
metrics = trainer.evaluate(eval_data)

print(f"SAE Quality Metrics:")
print(f"  Explained Variance: {metrics['explained_variance']:.3f} (target: > 0.85)")
print(f"  L0 Sparsity: {metrics['l0']:.1f} (target: ~64)")
print(f"  Dead Features: {metrics['dead_features']}/{sae.d_dict}")

## Phase 4: Contrastive Reasoning Feature Discovery 🔬

**This is our novel contribution.** We identify which SAE features encode reasoning behavior by comparing:
- Feature activations during **chain-of-thought** prompts
- Feature activations during **direct answer** prompts

Features with significantly higher CoT activation are **reasoning-associated**.

In [ ]:
from src.analysis.contrastive_features import ContrastiveFeatureDiscovery

discovery = ContrastiveFeatureDiscovery(
    sae=sae,
    significance_alpha=0.05,
    correction='bonferroni',
    min_effect_size=0.2,
)

# Load activations at middle denoising timestep
MIDDLE_TIMESTEP = 14
cot_acts = collector.load_activations('cot_activations', PRIMARY_LAYER, MIDDLE_TIMESTEP)
direct_acts = collector.load_activations('direct_activations', PRIMARY_LAYER, MIDDLE_TIMESTEP)

print(f"CoT activations: {cot_acts.shape}")
print(f"Direct activations: {direct_acts.shape}")

# Run contrastive analysis
print("\n🔬 Running contrastive analysis...")
results = discovery.analyze(cot_acts, direct_acts)

reasoning_features = discovery.get_top_reasoning_features(n=50)
print(f"\n✅ Found {results['n_reasoning']} reasoning features")
print(f"   Found {results['n_anti_reasoning']} anti-reasoning features")
print(f"   Top 10: {reasoning_features[:10]}")

# Save results
discovery.save(os.path.join(SAVE_DIR, 'feature_discovery'))

In [ ]:
# Visualize top reasoning features
from src.analysis.feature_visualizer import plot_differential_heatmap

plot_differential_heatmap(
    results, top_n=25,
    save_dir=os.path.join(SAVE_DIR, 'figures'),
)

# Display inline
from IPython.display import Image, display
fig_path = os.path.join(SAVE_DIR, 'figures', 'fig1_differential_heatmap.png')
if os.path.exists(fig_path):
    display(Image(fig_path))

## Phase 5: Steering Experiments 🎯

We now test whether amplifying these reasoning features during DLM denoising actually improves mathematical problem-solving.

**Experiments:**
- **E1**: Baseline (no steering)
- **E2**: Positive steering (α=2.0) — amplify reasoning features
- **E3**: Negative steering (α=-2.0) — suppress reasoning features
- **E4**: Random control — steer with random features (should ≈ baseline)
- **E5**: Alpha sweep — find optimal steering strength

In [ ]:
from src.steering.diffusion_steerer import SteeringExperiment
from src.analysis.steering_evaluator import (
    evaluate_experiment, evaluate_alpha_sweep,
    print_evaluation_summary, save_evaluation,
)

experiment = SteeringExperiment(
    dlm_wrapper=dlm,
    sae=sae,
    reasoning_features=reasoning_features[:50],
    target_layer=PRIMARY_LAYER,
)

# Use evaluation set
eval_prompts = eval_set.get_cot_prompts()[:30]  # 30 problems for speed
GEN_KWARGS = {'max_new_tokens': 128, 'num_steps': 30, 'temperature': 0.8}

all_evaluations = {}

In [ ]:
# E1: Baseline
print("[E1] Baseline (no steering)...")
baseline_results = experiment.run_baseline(eval_prompts, **GEN_KWARGS)
print(f"  ✅ Generated {len(baseline_results)} outputs")

In [ ]:
# E2: Positive steering
print("[E2] Positive steering (α=2.0)...")
steered_results = experiment.run_steered(
    eval_prompts, alpha=2.0, condition_name='positive_2.0', **GEN_KWARGS
)
eval_e2 = evaluate_experiment(baseline_results, steered_results, 'Positive (α=2.0)')
print_evaluation_summary(eval_e2)
all_evaluations['positive_2.0'] = eval_e2

In [ ]:
# E3: Negative steering
print("[E3] Negative steering (α=-2.0)...")
neg_results = experiment.run_steered(
    eval_prompts, alpha=-2.0, condition_name='negative_-2.0', **GEN_KWARGS
)
eval_e3 = evaluate_experiment(baseline_results, neg_results, 'Negative (α=-2.0)')
print_evaluation_summary(eval_e3)
all_evaluations['negative_-2.0'] = eval_e3

In [ ]:
# E4: Random control
print("[E4] Random feature control...")
random_results = experiment.run_random_control(
    eval_prompts[:15], n_features=50, n_random_sets=3, alpha=2.0, **GEN_KWARGS
)
eval_e4 = evaluate_experiment(baseline_results[:15], random_results[:15], 'Random Control')
print_evaluation_summary(eval_e4)
all_evaluations['random_control'] = eval_e4

In [ ]:
# E5: Alpha sweep
print("[E5] Alpha sweep...")
alpha_results = experiment.run_alpha_sweep(
    eval_prompts[:20],
    alpha_values=[0.5, 1.0, 2.0, 5.0, 10.0],
    **GEN_KWARGS,
)
alpha_evals = evaluate_alpha_sweep(baseline_results[:20], alpha_results)
for ae in alpha_evals:
    all_evaluations[f"alpha_{ae['alpha']}"] = ae
    print(f"  α={ae['alpha']}: acc={ae['steered_accuracy']['accuracy']:.1%}, S={ae['steering_score']:.3f}")

In [ ]:
# Save all results
results_dir = os.path.join(SAVE_DIR, 'experiment_results')
save_evaluation(all_evaluations, results_dir)

with open(os.path.join(results_dir, 'baseline_generations.json'), 'w') as f:
    json.dump(baseline_results, f, indent=2, default=str)
with open(os.path.join(results_dir, 'steered_generations.json'), 'w') as f:
    json.dump(steered_results, f, indent=2, default=str)

print("\n✅ All experiment results saved to Drive")

## Phase 6: Visualization 📊

In [ ]:
from src.analysis.feature_visualizer import (
    plot_accuracy_vs_alpha, plot_generation_examples,
    plot_results_summary, setup_style,
)
from IPython.display import Image, display

FIGURES_DIR = os.path.join(SAVE_DIR, 'figures')

# Fig 2: Accuracy vs Alpha
if alpha_evals:
    baseline_acc = alpha_evals[0].get('baseline_accuracy', {}).get('accuracy', 0)
    plot_accuracy_vs_alpha(alpha_evals, baseline_acc, save_dir=FIGURES_DIR)
    display(Image(os.path.join(FIGURES_DIR, 'fig2_accuracy_vs_alpha.png')))

# Fig 4: Generation examples
n = min(4, len(baseline_results), len(steered_results))
plot_generation_examples(
    [r.get('generated', r.get('text', '')) for r in baseline_results[:n]],
    [r.get('generated', r.get('text', '')) for r in steered_results[:n]],
    [r.get('prompt', '')[:100] for r in baseline_results[:n]],
    n_examples=n, save_dir=FIGURES_DIR,
)
display(Image(os.path.join(FIGURES_DIR, 'fig4_generation_examples.png')))

# Fig 6: Results summary
plot_results_summary(all_evaluations, save_dir=FIGURES_DIR)
display(Image(os.path.join(FIGURES_DIR, 'fig6_results_summary.png')))

## 🎉 Results Summary

Print the final results table:

In [ ]:
import pandas as pd

rows = []
for name, e in all_evaluations.items():
    rows.append({
        'Condition': name,
        'Accuracy': f"{e.get('steered_accuracy', e.get('baseline_accuracy', {})).get('accuracy', 0):.1%}",
        'Δ Accuracy': f"{e.get('accuracy_delta', 0):+.1%}",
        'Reasoning Score': f"{e.get('mean_reasoning_score_steered', e.get('mean_reasoning_score_baseline', 0)):.3f}",
        'Steering Score': f"{e.get('steering_score', 0):.3f}",
        'Significant': '✓' if e.get('reasoning_significant', False) else '✗',
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)

# Save as CSV
df.to_csv(os.path.join(SAVE_DIR, 'final_results.csv'), index=False)
print(f"\nResults saved to {SAVE_DIR}/final_results.csv")

## 📋 Next Steps

1. **Download figures** from Google Drive → `dlm_steering/figures/`
2. **Update README.md** with actual results numbers
3. **Write technical report** based on these findings
4. **Create project website** showcasing results

---

*Built as an extension of DLM-Scope (ICLR 2026) for the HKUNLP internship application.*